# Task 1: Classifying Parkinson's vs control
Run this notebook to predict whether a person has Parkinson's disease (PD) from their gait features.

The structure of this notebook is as follows:
1. Data setup: loading features, labels, splits
2. Training helper: function to train models and save outputs
3. Majority baseline
4. Logistic regression
5. Random forest
6. Results: comparing the models
7. 1D-CNN (to be added)

TODO: hyper param tuning

## 1. Data setup

In [52]:
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

# build data/processed/ if it is missing (features.py needs it)
from data import txt_to_csv, raw_dir, out_dir
if not out_dir.exists():
    txt_to_csv(raw_dir, out_dir)

SPLITS_PATH = ROOT / "data" / "splits.json"

# output directory
OUT = ROOT / "presentation_material" / "task1"
(OUT / "predictions").mkdir(parents=True, exist_ok=True)
(OUT / "weights").mkdir(parents=True, exist_ok=True)

# import functions from models
from features import X, y
from src.models.logreg import make_logistic_regression
from src.models.rf import make_random_forest

In [53]:
# X = one row per subject with 5 gait features, y = 1 if Parkinson's, 0 if healthy control
# (both come from features.py, so no need to merge metadata here)
print("Subjects:", X.shape[0], "| Feature matrix:", X.shape, "| PD share:", round(y.mean(), 3))
display(X.head())

Subjects: 165 | Feature matrix: (165, 5) | PD share: 0.564


,stride_time,stride_cv,swing_cv,swing_asymmetry,peak_force
GaCo01,1.230322,0.042125,0.054223,0.013902,14.561747
GaCo02,1.103386,0.035667,0.037527,0.001486,15.233429
GaCo03,1.241191,0.049523,0.078598,0.103336,12.754554
GaCo04,1.249965,0.058603,0.064637,0.022956,15.401692
GaCo05,1.051882,0.039561,0.055866,0.013298,12.820625


In [54]:
with open(SPLITS_PATH) as f:
    assignment = json.load(f)["task1"]

test_subjects = set(assignment["test"])
folds = []
for fold in assignment["folds"]:
    train_ids, val_ids = set(fold["train"]), set(fold["val"])
    # Essential safety check: keep the held-out test subjects separate.
    assert train_ids.isdisjoint(val_ids | test_subjects)
    assert val_ids.isdisjoint(test_subjects)
    train_idx = np.flatnonzero(X.index.isin(train_ids))
    val_idx = np.flatnonzero(X.index.isin(val_ids))
    folds.append((train_idx, val_idx))

print("Using", len(folds), "saved validation folds;", len(test_subjects), "held-out test subjects untouched.")

Using 5 saved validation folds; 33 held-out test subjects untouched.


## 2. Training helper

Function to implement at every model to train the model and save the output.
For each fold it trains on the training subjects only, predicts the probability of PD for the validation subjects, and saves the predictions and the fitted model.

In [55]:
def train_and_save(model_name, make_model):
    for k, (tr, va) in enumerate(folds, start = 1):
        model = make_model() # new, unfitted model per fold
        model.fit(X.iloc[tr], y.iloc[tr])
        prob = model.predict_proba(X.iloc[va])[:, 1]   # probability of PD
        pd.DataFrame({
            "subject": X.index[va],
            "fold": k,
            "y_true": y.iloc[va].to_numpy(),
            "y_pred": (prob >= 0.5).astype(int),
            "y_prob": prob,
        }).to_csv(
            OUT / "predictions" / f"{model_name}_val_fold{k}.csv", index = False
        )
        # save fitted preprocessing steps and model together.
        joblib.dump(
            model, OUT / "weights" / f"{model_name}_val_fold{k}.joblib"
        )
        print(f"{model_name}: saved validation fold {k} ({len(va)} rows)")

    print(f"{model_name}: model train done")

## 3. Majority baseline
This is used as a reference point. About 56% of subjects are classified correctly and the ROC-AUC is 0.5.

In [56]:
# "prior" always predicts the training-fold majority class (PD) and gives a constant probability
train_and_save("majority_baseline", lambda: DummyClassifier(strategy="prior"))

majority_baseline: saved validation fold 1 (27 rows)
majority_baseline: saved validation fold 2 (27 rows)
majority_baseline: saved validation fold 3 (26 rows)
majority_baseline: saved validation fold 4 (26 rows)
majority_baseline: saved validation fold 5 (26 rows)
majority_baseline: model train done


## 4. Logistic Regression
Weighs each gait feature and turns the total into a probability of PD. Features are standardised first (see `src/models/logreg.py`).

In [57]:
train_and_save("logistic_regression", make_logistic_regression)

logistic_regression: saved validation fold 1 (27 rows)
logistic_regression: saved validation fold 2 (27 rows)
logistic_regression: saved validation fold 3 (26 rows)
logistic_regression: saved validation fold 4 (26 rows)
logistic_regression: saved validation fold 5 (26 rows)
logistic_regression: model train done


## 5. Random Forest

In [58]:
train_and_save("random_forest", lambda: make_random_forest(task="classification"))

random_forest: saved validation fold 1 (27 rows)
random_forest: saved validation fold 2 (27 rows)
random_forest: saved validation fold 3 (26 rows)
random_forest: saved validation fold 4 (26 rows)
random_forest: saved validation fold 5 (26 rows)
random_forest: model train done


## Tuning Helper

In [61]:
from sklearn.model_selection import ParameterGrid

def tune(make_model, param_grid):
    rows = []
    for params in ParameterGrid(param_grid):
        aucs = []
        for train_idx, val_idx in folds:
            model = make_model()
            model.set_params(**params)
            model.fit(X.iloc[train_idx], y.iloc[train_idx])
            aucs.append(roc_auc_score(y.iloc[val_idx], model.predict_proba(X.iloc[val_idx])[:, 1]))
        rows.append({**params, "mean_AUC": np.mean(aucs), "std_AUC": np.std(aucs, ddof=1)})
    best = max(rows, key=lambda r: r["mean_AUC"])
    best_params = {k: best[k] for k in param_grid}   # taken from the list, so None stays None
    table = pd.DataFrame(rows).sort_values("mean_AUC", ascending=False)
    return table, best_params

In [64]:
lr_table, lr_best = tune(make_logistic_regression, {"model__C": [0.01, 0.1, 1, 10, 100]})
display(lr_table)
print("Best:", lr_best)

,model__C,mean_AUC,std_AUC
2,1.00,0.768911,0.055851
3,10.00,0.764365,0.053389
4,100.00,0.764365,0.053389
0,0.01,0.749343,0.067708
1,0.10,0.748009,0.060093


Best: {'model__C': 1}


In [65]:
rf_table, rf_best = tune(lambda: make_random_forest(task = "classification"),
                         {"model__n_estimators": [100, 200],
                          "model__max_depth": [2, 3, 5, None],
                          "model__min_samples_leaf": [1, 2, 5, 10]})
display(rf_table)
print("Best:", rf_best)

,model__max_depth,model__min_samples_leaf,model__n_estimators,mean_AUC,std_AUC
13,3.0,5,200,0.754675,0.079687
19,5.0,2,200,0.747785,0.053507
29,NaN,5,200,0.747698,0.091166
7,2.0,10,200,0.747605,0.077306
2,2.0,2,100,0.747345,0.077316
12,3.0,5,100,0.746616,0.079369
1,2.0,1,200,0.746494,0.077602
3,2.0,2,200,0.746494,0.077602
11,3.0,2,200,0.745404,0.084562
21,5.0,5,200,0.745375,0.093030


Best: {'model__n_estimators': 200, 'model__max_depth': 3, 'model__min_samples_leaf': 5}


In [66]:
train_and_save("logistic_regression_tuned", lambda: make_logistic_regression().set_params(**lr_best))
train_and_save("random_forest_tuned",
               lambda: make_random_forest(task="classification").set_params(**rf_best))

logistic_regression_tuned: saved validation fold 1 (27 rows)
logistic_regression_tuned: saved validation fold 2 (27 rows)
logistic_regression_tuned: saved validation fold 3 (26 rows)
logistic_regression_tuned: saved validation fold 4 (26 rows)
logistic_regression_tuned: saved validation fold 5 (26 rows)
logistic_regression_tuned: model train done
random_forest_tuned: saved validation fold 1 (27 rows)
random_forest_tuned: saved validation fold 2 (27 rows)
random_forest_tuned: saved validation fold 3 (26 rows)
random_forest_tuned: saved validation fold 4 (26 rows)
random_forest_tuned: saved validation fold 5 (26 rows)
random_forest_tuned: model train done


## 6. 1D-CNN Classification

In [59]:
import tensorflow as tf
from data import load_metadata
from src.models.cnn1d import build_cnn

PROCESSED = ROOT / "data" / "processed"
TIME_STEPS = 1000        # 10 seconds at 100 Hz
N_WINDOWS = 1            # 1 = first 10 s only (same as Task 2); set to 4 for more training examples
SENSOR_COLS = [f"L{i}" for i in range(1, 9)] + [f"R{i}" for i in range(1, 9)]
weight = load_metadata().set_index("ID")["Weight (kg)"]

def load_signal(subject):
    walk = pd.read_csv(PROCESSED / f"{subject}_01.csv")
    signal = walk[SENSOR_COLS].iloc[:TIME_STEPS * N_WINDOWS].to_numpy(dtype=np.float32)
    if signal.shape != (TIME_STEPS * N_WINDOWS, 16) or not np.isfinite(signal).all():
        raise ValueError(f"Invalid CNN signal for {subject}: {signal.shape}")
    signal = signal / weight[subject]   # force per kg, so the model sees gait and not body weight
    return signal.reshape(N_WINDOWS, TIME_STEPS, 16)

signals = {subject: load_signal(subject) for subject in X.index}
print("windows per subject:", N_WINDOWS, "| one subject:", signals[X.index[0]].shape)

windows per subject: 1 | one subject: (1, 1000, 16)


In [60]:
tf.keras.utils.set_random_seed(42)
EPOCHS = 20
BATCH_SIZE = 16

for fold_number, (train_idx, val_idx) in enumerate(folds, start=1):
    train_subjects, val_subjects = list(X.index[train_idx]), list(X.index[val_idx])

    # training windows: every window carries its subject's label (1 = PD, 0 = control)
    train_windows = np.concatenate([signals[s] for s in train_subjects])
    train_labels = np.repeat(y.loc[train_subjects].to_numpy(), N_WINDOWS)

    # normalise each sensor channel using training windows only
    mean = train_windows.mean(axis=(0, 1), keepdims=True)
    std = np.maximum(train_windows.std(axis=(0, 1), keepdims=True), 1e-6)

    # create and train a new model for each fold
    model = build_cnn(input_shape=(TIME_STEPS, 16), task="classification")
    model.fit((train_windows - mean) / std, train_labels,
              epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1)

    # one probability per validation subject = average over their windows
    probabilities = np.array([model.predict((signals[s] - mean) / std, verbose=0).mean()
                              for s in val_subjects])

    # save predictions and model weights
    pd.DataFrame({"subject": val_subjects,
                  "fold": fold_number,
                  "y_true": y.loc[val_subjects].to_numpy(),
                  "y_pred": (probabilities >= 0.5).astype(int),
                  "y_prob": probabilities}).to_csv(
        OUT / "predictions" / f"cnn_1d_val_fold{fold_number}.csv", index=False)
    model.save(OUT / "weights" / f"cnn_1d_val_fold{fold_number}.keras")
    np.savez(OUT / "weights" / f"cnn_1d_val_fold{fold_number}_normalisation.npz",
             mean=mean, std=std, time_steps=TIME_STEPS, n_windows=N_WINDOWS,
             sensor_columns=np.array(SENSOR_COLS))
    print(f"Saved CNN fold {fold_number} ({len(val_subjects)} predictions)")
print("CNN-1D model training complete.")

Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 59ms/step - accuracy: 0.5048 - loss: 0.6878
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5619 - loss: 0.6603 
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6000 - loss: 0.6452 
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7143 - loss: 0.6207 
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7048 - loss: 0.5950 
Epoch 6/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7238 - loss: 0.5763 
Epoch 7/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7429 - loss: 0.5556 
Epoch 8/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7333 - loss: 0.5367 
Epoch 9/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7238 - loss: 0.5229 
Epoch 10/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7333 - loss: 0.5077 
Epoch 11/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7333 - loss: 0.4973 
Epoch 12/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7524 - loss: 0.4874 
E

In [67]:
cnn_rows = []
for lr in [0.001, 0.0001]:
    for ep in [10, 20, 30]:
        aucs = []
        for train_idx, val_idx in folds:
            tf.keras.utils.set_random_seed(42)
            tr_ids, va_ids = list(X.index[train_idx]), list(X.index[val_idx])
            tr_windows = np.concatenate([signals[s] for s in tr_ids])
            tr_labels = np.repeat(y.loc[tr_ids].to_numpy(), N_WINDOWS)
            mean = tr_windows.mean(axis=(0, 1), keepdims=True)
            std = np.maximum(tr_windows.std(axis=(0, 1), keepdims=True), 1e-6)
            model = build_cnn(input_shape=(TIME_STEPS, 16), task="classification", learning_rate=lr)
            model.fit((tr_windows - mean) / std, tr_labels, epochs=ep, batch_size=BATCH_SIZE, verbose=0)
            prob = [model.predict((signals[s] - mean) / std, verbose=0).mean() for s in va_ids]
            aucs.append(roc_auc_score(y.loc[va_ids], prob))
        cnn_rows.append({"learning_rate": lr, "epochs": ep,
                         "mean_AUC": np.mean(aucs), "std_AUC": np.std(aucs, ddof=1)})
display(pd.DataFrame(cnn_rows).sort_values("mean_AUC", ascending=False))

,learning_rate,epochs,mean_AUC,std_AUC
0,0.0010,10,0.776595,0.048113
1,0.0010,20,0.771097,0.027595
2,0.0010,30,0.764329,0.034681
5,0.0001,30,0.739221,0.059510
4,0.0001,20,0.727749,0.072059
3,0.0001,10,0.715808,0.080257
